# UMTAM Pruning Experiments: 7 GLUE Tasks

**Section 4.2: Pruning on Natural Language Understanding Tasks**

This notebook validates UMTAM's curvature-aware saliency scores by comparing:
- **Curvature-aware pruning**: Uses full saliency = (w - w₀)² × √(R ⊗ C)
- **Magnitude-only pruning**: Uses only (w - w₀)²

Tasks: RTE, MRPC, CoLA, SST-2, QNLI, QQP, MNLI

**Runtime estimate**: ~2-3 hours on Colab GPU (T4/L4)

## Setup & Installation

In [ ]:
!pip install -q transformers datasets accelerate scikit-learn matplotlib seaborn

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from transformers import (
    BertTokenizer, 
    BertForSequenceClassification,
    get_linear_schedule_with_warmup
)
from datasets import load_dataset
from tqdm.auto import tqdm
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, f1_score, matthews_corrcoef
import warnings
import copy
import gc
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## Task Configuration

In [ ]:
# GLUE task configurations
TASK_CONFIG = {
    'rte': {
        'num_labels': 2,
        'metric': 'accuracy',
        'text_fields': ('sentence1', 'sentence2'),
        'epochs': 5,
        'lr': 2e-5,
        'batch_size': 16,
        'max_samples': None  # Use all
    },
    'mrpc': {
        'num_labels': 2,
        'metric': 'f1',
        'text_fields': ('sentence1', 'sentence2'),
        'epochs': 5,
        'lr': 2e-5,
        'batch_size': 16,
        'max_samples': None
    },
    'cola': {
        'num_labels': 2,
        'metric': 'matthews',
        'text_fields': ('sentence', None),
        'epochs': 5,
        'lr': 2e-5,
        'batch_size': 16,
        'max_samples': None
    },
    'sst2': {
        'num_labels': 2,
        'metric': 'accuracy',
        'text_fields': ('sentence', None),
        'epochs': 3,
        'lr': 2e-5,
        'batch_size': 32,
        'max_samples': 10000  # Subset for speed
    },
    'qnli': {
        'num_labels': 2,
        'metric': 'accuracy',
        'text_fields': ('question', 'sentence'),
        'epochs': 3,
        'lr': 2e-5,
        'batch_size': 32,
        'max_samples': 10000
    },
    'qqp': {
        'num_labels': 2,
        'metric': 'f1',
        'text_fields': ('question1', 'question2'),
        'epochs': 3,
        'lr': 2e-5,
        'batch_size': 32,
        'max_samples': 15000
    },
    'mnli': {
        'num_labels': 3,
        'metric': 'accuracy',
        'text_fields': ('premise', 'hypothesis'),
        'epochs': 3,
        'lr': 2e-5,
        'batch_size': 32,
        'max_samples': 15000,
        'val_split': 'validation_matched'
    }
}

# Density levels to test
DENSITY_LEVELS = [1, 5, 10, 20, 40, 60, 90]

print(f"Tasks to evaluate: {list(TASK_CONFIG.keys())}")
print(f"Density levels: {DENSITY_LEVELS}%")

## Data Loading Functions

In [ ]:
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

def load_glue_task(task_name):
    """Load and preprocess a GLUE task."""
    config = TASK_CONFIG[task_name]
    
    # Load dataset
    dataset = load_dataset('glue', task_name)
    
    # Get text fields
    text1_field, text2_field = config['text_fields']
    
    def tokenize_fn(examples):
        if text2_field:
            return tokenizer(
                examples[text1_field],
                examples[text2_field],
                padding='max_length',
                truncation=True,
                max_length=128
            )
        else:
            return tokenizer(
                examples[text1_field],
                padding='max_length',
                truncation=True,
                max_length=128
            )
    
    # Tokenize
    tokenized = dataset.map(tokenize_fn, batched=True, remove_columns=dataset['train'].column_names[:-1])
    tokenized = tokenized.rename_column('label', 'labels')
    tokenized.set_format('torch', columns=['input_ids', 'attention_mask', 'labels'])
    
    # Get splits
    train_data = tokenized['train']
    val_split = config.get('val_split', 'validation')
    val_data = tokenized[val_split]
    
    # Subsample if needed
    max_samples = config['max_samples']
    if max_samples and len(train_data) > max_samples:
        train_data = train_data.shuffle(seed=42).select(range(max_samples))
    
    return train_data, val_data

def compute_metric(preds, labels, metric_name):
    """Compute task-specific metric."""
    if metric_name == 'accuracy':
        return accuracy_score(labels, preds)
    elif metric_name == 'f1':
        return f1_score(labels, preds, average='binary')
    elif metric_name == 'matthews':
        return matthews_corrcoef(labels, preds)
    else:
        raise ValueError(f"Unknown metric: {metric_name}")

print("✓ Data loading functions defined")

## UMTAM Training with Curvature Tracking

In [ ]:
def train_with_curvature_tracking(task_name):
    """Train BERT on a GLUE task while tracking curvature statistics."""
    config = TASK_CONFIG[task_name]
    print(f"\n{'='*60}")
    print(f"Training on {task_name.upper()}")
    print(f"{'='*60}")
    
    # Load data
    train_data, val_data = load_glue_task(task_name)
    print(f"Train samples: {len(train_data)}, Val samples: {len(val_data)}")
    
    train_loader = DataLoader(train_data, batch_size=config['batch_size'], shuffle=True)
    val_loader = DataLoader(val_data, batch_size=config['batch_size'])
    
    # Initialize model
    model = BertForSequenceClassification.from_pretrained(
        'bert-base-uncased',
        num_labels=config['num_labels']
    ).to(device)
    
    # Store initial weights
    w0_dict = {}
    curvature_dict = {}
    
    for name, param in model.named_parameters():
        if param.requires_grad and param.dim() == 2:
            w0_dict[name] = param.data.clone().cpu()
            m, n = param.shape
            curvature_dict[name] = {
                'R': torch.zeros(m, device='cpu'),
                'C': torch.zeros(n, device='cpu')
            }
    
    # Optimizer and scheduler
    optimizer = torch.optim.AdamW(model.parameters(), lr=config['lr'], weight_decay=0.01)
    total_steps = len(train_loader) * config['epochs']
    scheduler = get_linear_schedule_with_warmup(
        optimizer, 
        num_warmup_steps=int(0.1 * total_steps),
        num_training_steps=total_steps
    )
    
    # Training loop
    beta2 = 0.999
    step = 0
    
    for epoch in range(config['epochs']):
        model.train()
        total_loss = 0
        
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{config['epochs']}")
        for batch in pbar:
            batch = {k: v.to(device) for k, v in batch.items()}
            
            outputs = model(**batch)
            loss = outputs.loss
            
            optimizer.zero_grad()
            loss.backward()
            
            # Update curvature statistics (EMA of squared gradients)
            with torch.no_grad():
                for name, param in model.named_parameters():
                    if name in curvature_dict and param.grad is not None:
                        g = param.grad
                        g2 = g ** 2
                        
                        # Row-wise: mean over columns
                        R_new = g2.mean(dim=1).cpu()
                        # Column-wise: mean over rows  
                        C_new = g2.mean(dim=0).cpu()
                        
                        # EMA update
                        curvature_dict[name]['R'] = beta2 * curvature_dict[name]['R'] + (1 - beta2) * R_new
                        curvature_dict[name]['C'] = beta2 * curvature_dict[name]['C'] + (1 - beta2) * C_new
            
            optimizer.step()
            scheduler.step()
            step += 1
            
            total_loss += loss.item()
            pbar.set_postfix({'loss': f'{loss.item():.4f}'})
        
        avg_loss = total_loss / len(train_loader)
        print(f"  Epoch {epoch+1} average loss: {avg_loss:.4f}")
    
    # Compute saliency scores
    print("Computing saliency scores...")
    saliency_dict = {}
    
    for name in curvature_dict.keys():
        param = dict(model.named_parameters())[name]
        w0 = w0_dict[name].to(device)
        
        # Parameter change: (w - w0)²
        param_change = (param.data - w0) ** 2
        
        # Curvature approximation: sqrt(R ⊗ C)
        R = curvature_dict[name]['R'].to(device).clamp(min=1e-8)
        C = curvature_dict[name]['C'].to(device).clamp(min=1e-8)
        curvature_approx = torch.sqrt(R.unsqueeze(1) * C.unsqueeze(0))
        
        # Saliency = parameter_change × curvature
        saliency_dict[name] = (param_change * curvature_approx).cpu()
    
    # Evaluate unpruned model
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in val_loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = model(**batch)
            preds = outputs.logits.argmax(dim=-1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(batch['labels'].cpu().numpy())
    
    baseline_score = compute_metric(all_preds, all_labels, config['metric'])
    print(f"✓ Baseline {config['metric']}: {baseline_score:.4f}")
    
    return model, w0_dict, saliency_dict, curvature_dict, val_loader, config, baseline_score

print("✓ Training function defined")

## Pruning Functions

In [ ]:
def apply_pruning(model, w0_dict, saliency_dict, density_pct, use_curvature=True):
    """Apply pruning and return a pruned copy of the model."""
    pruned_model = copy.deepcopy(model)
    
    # Collect all scores
    all_scores = []
    for name in saliency_dict.keys():
        if use_curvature:
            scores = saliency_dict[name].flatten()
        else:
            # Magnitude-only: just parameter change
            param = dict(pruned_model.named_parameters())[name]
            w0 = w0_dict[name].to(device)
            scores = ((param.data - w0) ** 2).cpu().flatten()
        all_scores.append(scores)
    
    all_scores = torch.cat(all_scores)
    
    # Determine threshold
    k = int(len(all_scores) * (density_pct / 100.0))
    k = max(1, min(k, len(all_scores) - 1))
    threshold = torch.kthvalue(all_scores, len(all_scores) - k).values
    
    # Apply grafting
    with torch.no_grad():
        for name, param in pruned_model.named_parameters():
            if name in saliency_dict:
                w0 = w0_dict[name].to(device)
                
                if use_curvature:
                    scores = saliency_dict[name].to(device)
                else:
                    scores = (param.data - w0) ** 2
                
                mask = (scores > threshold).float()
                param.data = mask * param.data + (1 - mask) * w0
    
    return pruned_model

def evaluate_model(model, val_loader, metric_name):
    """Evaluate model and return metric score."""
    model.eval()
    all_preds, all_labels = [], []
    
    with torch.no_grad():
        for batch in val_loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = model(**batch)
            preds = outputs.logits.argmax(dim=-1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(batch['labels'].cpu().numpy())
    
    return compute_metric(all_preds, all_labels, metric_name)

print("✓ Pruning functions defined")

## Run Pruning Experiments

In [ ]:
# Store all results
all_results = {}

for task_name in TASK_CONFIG.keys():
    # Train model with curvature tracking
    model, w0_dict, saliency_dict, curvature_dict, val_loader, config, baseline_score = \
        train_with_curvature_tracking(task_name)
    
    task_results = {
        'baseline': baseline_score,
        'metric': config['metric'],
        'curvature': {},
        'magnitude': {}
    }
    
    print(f"\nEvaluating pruning at different densities...")
    
    for density in DENSITY_LEVELS:
        # Curvature-aware pruning
        pruned_curv = apply_pruning(model, w0_dict, saliency_dict, density, use_curvature=True)
        score_curv = evaluate_model(pruned_curv, val_loader, config['metric'])
        task_results['curvature'][density] = score_curv
        
        # Magnitude-only pruning
        pruned_mag = apply_pruning(model, w0_dict, saliency_dict, density, use_curvature=False)
        score_mag = evaluate_model(pruned_mag, val_loader, config['metric'])
        task_results['magnitude'][density] = score_mag
        
        improvement = ((score_curv - score_mag) / max(abs(score_mag), 1e-6)) * 100
        print(f"  {density:2d}% density: Curvature={score_curv:.4f}, Magnitude={score_mag:.4f}, Δ={improvement:+.1f}%")
        
        # Clean up
        del pruned_curv, pruned_mag
        torch.cuda.empty_cache()
    
    all_results[task_name] = task_results
    
    # Clean up task
    del model, w0_dict, saliency_dict, curvature_dict
    gc.collect()
    torch.cuda.empty_cache()

print("\n" + "="*60)
print("All experiments completed!")
print("="*60)

## Results Summary Table

In [ ]:
import pandas as pd

# Create summary DataFrame
rows = []
for task_name, results in all_results.items():
    metric = results['metric']
    baseline = results['baseline']
    
    for density in DENSITY_LEVELS:
        curv_score = results['curvature'][density]
        mag_score = results['magnitude'][density]
        
        # Compute improvement
        if mag_score != 0:
            improvement = ((curv_score - mag_score) / abs(mag_score)) * 100
        else:
            improvement = 0
        
        rows.append({
            'Task': task_name.upper(),
            'Metric': metric,
            'Baseline': baseline,
            'Density (%)': density,
            'Curvature': curv_score,
            'Magnitude': mag_score,
            'Δ (%)': improvement
        })

df = pd.DataFrame(rows)

# Display pivot table
print("\n" + "="*80)
print("PRUNING RESULTS: Curvature-Aware vs Magnitude-Only")
print("="*80)

for task in all_results.keys():
    task_df = df[df['Task'] == task.upper()]
    print(f"\n{task.upper()} (Metric: {task_df['Metric'].iloc[0]}, Baseline: {task_df['Baseline'].iloc[0]:.4f})")
    print("-" * 60)
    print(f"{'Density':>10} {'Curvature':>12} {'Magnitude':>12} {'Improvement':>12}")
    for _, row in task_df.iterrows():
        print(f"{row['Density (%)']:>9}% {row['Curvature']:>12.4f} {row['Magnitude']:>12.4f} {row['Δ (%)']:>+11.1f}%")

## Generate LaTeX Table

In [ ]:
# Generate LaTeX table for paper
latex_output = r"""
\begin{table*}[t]
\centering
\caption{Pruning performance comparison: Curvature-aware (UMTAM) vs.\ Magnitude-only saliency across seven GLUE tasks at varying density levels. Higher values indicate better performance. Bold indicates the better method at each density level.}
\label{tab:pruning_7tasks}
\begin{tabular}{ll|ccccccc}
\toprule
\textbf{Task} & \textbf{Method} & \textbf{1\%} & \textbf{5\%} & \textbf{10\%} & \textbf{20\%} & \textbf{40\%} & \textbf{60\%} & \textbf{90\%} \\
\midrule
"""

for task_name in all_results.keys():
    results = all_results[task_name]
    
    # Curvature row
    curv_vals = [results['curvature'][d] for d in DENSITY_LEVELS]
    mag_vals = [results['magnitude'][d] for d in DENSITY_LEVELS]
    
    curv_strs = []
    mag_strs = []
    
    for c, m in zip(curv_vals, mag_vals):
        if c > m:
            curv_strs.append(f"\\textbf{{{c:.3f}}}")
            mag_strs.append(f"{m:.3f}")
        elif m > c:
            curv_strs.append(f"{c:.3f}")
            mag_strs.append(f"\\textbf{{{m:.3f}}}")
        else:
            curv_strs.append(f"{c:.3f}")
            mag_strs.append(f"{m:.3f}")
    
    latex_output += f"{task_name.upper()} & Curvature & {' & '.join(curv_strs)} \\\\\n"
    latex_output += f" & Magnitude & {' & '.join(mag_strs)} \\\\\n"
    latex_output += "\\midrule\n"

latex_output = latex_output.rstrip("\\midrule\n") + """
\\bottomrule
\\end{tabular}
\\end{table*}
"""

print(latex_output)

## Visualization

In [ ]:
# Create visualization
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

tasks = list(all_results.keys())

for idx, task_name in enumerate(tasks):
    ax = axes[idx]
    results = all_results[task_name]
    
    curv_scores = [results['curvature'][d] for d in DENSITY_LEVELS]
    mag_scores = [results['magnitude'][d] for d in DENSITY_LEVELS]
    
    ax.plot(DENSITY_LEVELS, curv_scores, 'b-o', label='Curvature-aware', linewidth=2, markersize=6)
    ax.plot(DENSITY_LEVELS, mag_scores, 'r--s', label='Magnitude-only', linewidth=2, markersize=6)
    ax.axhline(y=results['baseline'], color='green', linestyle=':', label=f'Baseline', alpha=0.7)
    
    ax.set_xlabel('Density (%)')
    ax.set_ylabel(results['metric'].capitalize())
    ax.set_title(f"{task_name.upper()}")
    ax.legend(loc='lower right', fontsize=8)
    ax.grid(True, alpha=0.3)
    ax.set_xlim([0, 100])

# Hide unused subplot
if len(tasks) < 8:
    axes[7].axis('off')

plt.suptitle('Curvature-Aware vs Magnitude-Only Pruning on GLUE Tasks', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('pruning_7tasks.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n✓ Figure saved as 'pruning_7tasks.png'")

## Average Improvement Summary

In [ ]:
# Compute average improvement at each density level
print("\n" + "="*60)
print("AVERAGE IMPROVEMENT: Curvature over Magnitude")
print("="*60)

for density in DENSITY_LEVELS:
    improvements = []
    for task_name, results in all_results.items():
        c = results['curvature'][density]
        m = results['magnitude'][density]
        if m != 0:
            imp = ((c - m) / abs(m)) * 100
            improvements.append(imp)
    
    avg_imp = np.mean(improvements)
    print(f"  {density:2d}% density: Average improvement = {avg_imp:+.2f}%")

# Overall wins
curv_wins = 0
mag_wins = 0
ties = 0

for task_name, results in all_results.items():
    for density in DENSITY_LEVELS:
        c = results['curvature'][density]
        m = results['magnitude'][density]
        if c > m:
            curv_wins += 1
        elif m > c:
            mag_wins += 1
        else:
            ties += 1

total = curv_wins + mag_wins + ties
print(f"\nOverall: Curvature wins {curv_wins}/{total} ({100*curv_wins/total:.1f}%), "
      f"Magnitude wins {mag_wins}/{total} ({100*mag_wins/total:.1f}%), "
      f"Ties {ties}/{total}")

## Save Results

In [ ]:
import json

# Save results to JSON
with open('pruning_7tasks_results.json', 'w') as f:
    json.dump(all_results, f, indent=2)

# Save DataFrame to CSV
df.to_csv('pruning_7tasks_results.csv', index=False)

print("✓ Results saved to:")
print("  - pruning_7tasks_results.json")
print("  - pruning_7tasks_results.csv")
print("  - pruning_7tasks.png")